# Build Team Form - Silver Layer

Calculates last 5 matches rolling statistics for each team.

## Target Schema
* **team_id** (INT, PK)
* **team_name** (STRING)
* **last5_wins** (INT)
* **last5_draws** (INT)
* **last5_losses** (INT)
* **last5_goals_scored** (INT)
* **last5_goals_conceded** (INT)
* **form_string** (STRING) - e.g., "W W D L W"
* **last_updated** (TIMESTAMP)

## Process
1. Read matches from bronze layer
2. Calculate match outcomes for each team
3. Window over last 5 matches per team
4. Aggregate statistics
5. Build form string

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import DoubleType, StringType
from config.paths import MATCHES_BRONZE, EVENTS_BRONZE

# Initialize Spark session
spark = SparkSession.builder.appName("BuildTeamForm").getOrCreate()

print("=" * 80)
print("Building Team Form - Silver Layer")
print("=" * 80)

In [0]:
# Read matches from bronze layer
print("\n[1/6] Reading matches from bronze layer...")
df_matches_raw = spark.read.format("delta").load(MATCHES_BRONZE)
print(f"   Total rows loaded: {df_matches_raw.count():,}")

# Deduplicate matches (bronze has duplicate rows)
df_matches = df_matches_raw.dropDuplicates(["match_id"])
print(f"   Unique matches after deduplication: {df_matches.count():,}")

# Show sample
print("\nSample matches:")
display(df_matches.limit(2))

In [0]:
# Create a union of home and away team perspectives
print("\n[2/6] Calculating match outcomes for each team...")

# Home team perspective
df_home = df_matches.select(
    F.col("match_id"),
    F.col("match_date"),
    F.col("home_team.home_team_id").alias("team_id"),
    F.col("home_team.home_team_name").alias("team_name"),
    F.col("home_score").alias("goals_scored"),
    F.col("away_score").alias("goals_conceded"),
    F.when(F.col("home_score") > F.col("away_score"), "W")
     .when(F.col("home_score") == F.col("away_score"), "D")
     .otherwise("L").alias("result")
)

# Away team perspective
df_away = df_matches.select(
    F.col("match_id"),
    F.col("match_date"),
    F.col("away_team.away_team_id").alias("team_id"),
    F.col("away_team.away_team_name").alias("team_name"),
    F.col("away_score").alias("goals_scored"),
    F.col("home_score").alias("goals_conceded"),
    F.when(F.col("away_score") > F.col("home_score"), "W")
     .when(F.col("away_score") == F.col("home_score"), "D")
     .otherwise("L").alias("result")
)

# Union both perspectives
df_team_matches = df_home.union(df_away)

print(f"   Total team-match records: {df_team_matches.count():,}")
print("\nSample team matches:")
display(df_team_matches.orderBy("match_date", "team_name").limit(6))

In [0]:
# Define window to get last 5 matches per team (or all available if fewer)
print("\n[3/6] Windowing over last 5 matches per team...")

window_spec = Window.partitionBy("team_id").orderBy(F.desc("match_date")).rowsBetween(Window.currentRow, 4)

# Calculate rolling aggregates over last 5 matches (or all available)
df_rolling = df_team_matches.withColumn(
    "matches_in_window",
    F.count("*").over(window_spec)
).withColumn(
    "last5_wins",
    F.sum(F.when(F.col("result") == "W", 1).otherwise(0)).over(window_spec)
).withColumn(
    "last5_draws",
    F.sum(F.when(F.col("result") == "D", 1).otherwise(0)).over(window_spec)
).withColumn(
    "last5_losses",
    F.sum(F.when(F.col("result") == "L", 1).otherwise(0)).over(window_spec)
).withColumn(
    "last5_goals_scored",
    F.sum("goals_scored").over(window_spec)
).withColumn(
    "last5_goals_conceded",
    F.sum("goals_conceded").over(window_spec)
)

print("   Rolling statistics calculated ✓")

In [0]:
# Get the most recent record for each team (latest match date)
print("\n[4/6] Getting most recent form for each team...")

window_latest = Window.partitionBy("team_id").orderBy(F.desc("match_date"))

df_latest_form = df_rolling.withColumn(
    "row_num",
    F.row_number().over(window_latest)
).filter(
    F.col("row_num") == 1
).drop("row_num", "matches_in_window")

print(f"   Teams with form data: {df_latest_form.count():,}")

In [0]:
# Build form string by collecting last 5 results (or all available)
print("\n[5/6] Building form strings...")

# Window to collect last 5 results in order (or all available if fewer)
window_form = Window.partitionBy("team_id").orderBy(F.desc("match_date")).rowsBetween(0, 4)

# Collect results into array
df_with_form_array = df_team_matches.withColumn(
    "result_array",
    F.collect_list("result").over(window_form)
).withColumn(
    "array_size",
    F.size("result_array")
)

# Get latest record per team and build form string
df_form_strings = df_with_form_array.withColumn(
    "row_num",
    F.row_number().over(Window.partitionBy("team_id").orderBy(F.desc("match_date")))
).filter(
    F.col("row_num") == 1
).withColumn(
    "form_string",
    F.concat_ws(" ", F.col("result_array"))
).select(
    "team_id",
    "form_string"
)

print("   Form strings created ✓")

In [0]:
# Join form strings with aggregated stats
print("\n[6/6] Creating final dataset...")

df_final = df_latest_form.join(
    df_form_strings,
    on="team_id",
    how="left"
).withColumn(
    "last_updated",
    F.current_timestamp()
).select(
    F.col("team_id").cast("int"),
    F.col("team_name"),
    F.col("last5_wins").cast("int"),
    F.col("last5_draws").cast("int"),
    F.col("last5_losses").cast("int"),
    F.col("last5_goals_scored").cast("int"),
    F.col("last5_goals_conceded").cast("int"),
    F.col("form_string"),
    F.col("last_updated")
)

print(f"   Final dataset: {df_final.count():,} teams")
print("\nSample team form data:")
df_final.orderBy(F.desc("last5_wins")).show(10, truncate=False)

# Write to silver layer
print("\n[OUTPUT] Writing to matchpulse.silver.team_form...")
df_final.write.mode("overwrite").saveAsTable("matchpulse.silver.team_form")
print("✓ Successfully wrote to matchpulse.silver.team_form")

print("\n" + "=" * 80)
print("Build Complete")
print("=" * 80)

In [0]:
%sql
-- Verify the table was created and query sample data
SELECT 
    team_name,
    last5_wins,
    last5_draws,
    last5_losses,
    last5_goals_scored,
    last5_goals_conceded,
    form_string
FROM matchpulse.silver.team_form
ORDER BY last5_wins DESC, last5_goals_scored DESC;